In [1]:
import requests
from datetime import datetime, timedelta
import random
import pickle
import json
import os
from time import sleep
import pandas as pd

        

In [2]:
dict_character = {}

def get_all_characters():
    for _id in range(1, 827):
        endpoint = f"https://rickandmortyapi.com/api/character/{_id}"
        response = requests.get(endpoint)
        
        dict_character[endpoint] = response.json()["name"]
get_all_characters()
dict_character

def complete_data(dataframe):
    total_data = []
    if dataframe == "character":
        pages = 42
    elif dataframe == "episode":
        pages = 3
    elif dataframe == "location":
        pages = 7
    for page in range(pages):
        endpoint = f"https://rickandmortyapi.com/api/{dataframe}/?page={page+1}"
        response = requests.get(endpoint)
        data = response.json()["results"]
        total_data.extend(data)
    return total_data

In [3]:




def fetch_data():
    
    datos_char = complete_data("character")
    datos_loc = complete_data("location")
    datos_epi = complete_data("episode")
    data = (datos_char,datos_loc,datos_epi)

    return data

def fix_col_dict(data_frame: pd.DataFrame, nombre_col):
    
    datos =[]
    
    for fila in data_frame[nombre_col]:
        datos.append(fila["name"])
        
    data_frame[f"{nombre_col}_nueva"] = datos
    
    # Borramos la columna y la devolvemos
    return data_frame.drop(nombre_col, axis=1)


def fix_url(dataframe, nombre_columna):
    
    lista_de_celdas = []
    
    for fila in dataframe[nombre_columna]:
        lista_de_nombres = []
        
        for url in fila:
            lista_de_nombres.append(dict_character[url])
        
        lista_de_celdas.append(lista_de_nombres)
        
    dataframe[nombre_columna] = lista_de_celdas

def trasnform_data(total_data):
    
    data_characters, data_locations, data_episodes = total_data
    
    df_characters = pd.DataFrame(data_characters)
    
    df_characters = fix_col_dict(df_characters, "origin")
    df_characters = fix_col_dict(df_characters, "location")
    
    df_characters = df_characters.drop(["image","episode","url", "created"],axis = 1)
    
    df_locations = pd.DataFrame(data_locations)
    fix_url(df_locations, "residents")
    df_locations = df_locations.drop(["url", "created"],axis = 1)
    
    df_episodes = pd.DataFrame(data_episodes)
    fix_url(df_episodes, "characters")
    df_episodes = df_episodes.drop(["url", "created"],axis = 1)
    
    return df_characters, df_locations, df_episodes



            


In [5]:
#EJECUCION 
data = fetch_data()
df_characters, df_locations, df_episodes = trasnform_data(data)

In [6]:
df_characters


,id,name,status,species,type,gender,origin_nueva,location_nueva
0,1,Rick Sanchez,Alive,Human,,Male,Earth (C-137),Citadel of Ricks
1,2,Morty Smith,Alive,Human,,Male,unknown,Citadel of Ricks
2,3,Summer Smith,Alive,Human,,Female,Earth (Replacement Dimension),Earth (Replacement Dimension)
3,4,Beth Smith,Alive,Human,,Female,Earth (Replacement Dimension),Earth (Replacement Dimension)
4,5,Jerry Smith,Alive,Human,,Male,Earth (Replacement Dimension),Earth (Replacement Dimension)
...,...,...,...,...,...,...,...,...
821,822,Young Jerry,unknown,Human,,Male,Earth (Unknown dimension),Earth (Unknown dimension)
822,823,Young Beth,unknown,Human,,Female,Earth (Unknown dimension),Earth (Unknown dimension)
823,824,Young Beth,unknown,Human,,Female,Earth (Unknown dimension),Earth (Unknown dimension)
824,825,Young Jerry,unknown,Human,,Male,Earth (Unknown dimension),Earth (Unknown dimension)


In [7]:
df_episodes

,id,name,air_date,episode,characters
0,1,Pilot,"December 2, 2013",S01E01,"[Rick Sanchez, Morty Smith, Bepisian, Beth Smi..."
1,2,Lawnmower Dog,"December 9, 2013",S01E02,"[Rick Sanchez, Morty Smith, Beth Smith, Bill, ..."
2,3,Anatomy Park,"December 16, 2013",S01E03,"[Rick Sanchez, Morty Smith, Alexander, Annie, ..."
3,4,M. Night Shaym-Aliens!,"January 13, 2014",S01E04,"[Rick Sanchez, Morty Smith, Beth Smith, Cynthi..."
4,5,Meeseeks and Destroy,"January 20, 2014",S01E05,"[Rick Sanchez, Morty Smith, Beth Smith, Big Bo..."
5,6,Rick Potion #9,"January 27, 2014",S01E06,"[Rick Sanchez, Morty Smith, Summer Smith, Beth..."
6,7,Raising Gazorpazorp,"March 10, 2014",S01E07,"[Rick Sanchez, Morty Smith, Summer Smith, Beth..."
7,8,Rixty Minutes,"March 17, 2014",S01E08,"[Rick Sanchez, Morty Smith, Summer Smith, Beth..."
8,9,Something Ricked This Way Comes,"March 24, 2014",S01E09,"[Rick Sanchez, Morty Smith, Summer Smith, Beth..."
9,10,Close Rick-counters of the Rick Kind,"April 7, 2014",S01E10,"[Rick Sanchez, Morty Smith, Summer Smith, Beth..."


In [8]:
df_locations


,id,name,type,dimension,residents
0,1,Earth (C-137),Planet,Dimension C-137,"[Beth Smith, Bill, Conroy, Cronenberg Rick, Cr..."
1,2,Abadango,Cluster,unknown,[Abadango Cluster Princess]
2,3,Citadel of Ricks,Space station,unknown,"[Adjudicator Rick, Alien Morty, Alien Rick, An..."
3,4,Worldender's lair,Planet,unknown,"[Alan Rails, Crocubot, Logic, Million Ants, Su..."
4,5,Anatomy Park,Microverse,Dimension C-137,"[Alexander, Annie, Tuberculosis, Gonorrhea, He..."
...,...,...,...,...,...
121,122,Avian Planet,Planet,Replacement Dimension,"[Alien Crow, Alien Crow]"
122,123,Normal Size Bug Dimension,Dimension,,"[Palicki, Sarge]"
123,124,Slartivart,Planet,Replacement Dimension,[Slartivartian]
124,125,Rick and Two Crows Planet,Planet,Replacement Dimension,"[Pussifer, Crow Scare, Two Crows]"
